# Stage 11 decoder-side fixes — KenLM / length-norm beam / joint scoring

**No retraining.**  Operates on `stage11_best.pt` + the paper-split landmark cache.

Per the prompt's §4, four experiments in strict priority order, each with **val hyperparameter sweeps + a single test evaluation**:

| Experiment | Decoder | Hyperparams swept on val |
|---|---|---|
| 9b | CTC + LM prefix beam search | `alpha` (LM weight), `beta` (length bonus) |
| B1 | Attention beam with length norm | `beam_width`, `length_alpha` |
| B2 | Joint CTC + attention beam | `alpha_ctc`, `alpha_attn` (with `alpha_lm=0`) |
| B3 | KenLM + B1 + B2 combined | small 4-dim grid initialised from 9b/B1/B2 winners |

## Test-eval discipline (non-negotiable)

The eval script writes a marker file per (mode, hparams) hash to `/kaggle/working/logs/`.  Re-running the SAME config raises RuntimeError.  Per §7: 4 test evals total (one per experiment), each frozen on val-best.  Never look at test CER before frozen.

## Stage 11 baseline (for delta tracking)
- `test_overall_cer = 0.4498`
- `test_lex_cer     = 0.4348`
- `test_nonlex_cer  = 0.5449`
- `test_cer[1-4]    = 0.309`, `[13+] = 0.662`  ← length-driven gap

## Wall-clock
~3 GPU-hours total on Kaggle T4: ~30 min per val sweep × 4 + ~5 min per test eval × 4.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance scipy --quiet
import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate landmark cache + Stage 11 checkpoint

In [ ]:
import os, glob, shutil, json

def _find(pat):
    m = (glob.glob(f'/kaggle/working/**/{pat}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
    return m[0] if m else None

CACHE_ROOT = _find('landmark_cache_122')
CKPT       = _find('stage11_best.pt')
assert CACHE_ROOT, 'Attach the wita-full-english-landmark-cache dataset.'
assert CKPT,       'Attach the Stage 11 training kernel output (stage11_best.pt).'
print(f'cache_root : {CACHE_ROOT}')
print(f'checkpoint : {CKPT}')

LOG_DIR = '/kaggle/working/logs'
LM_DIR  = '/kaggle/working/lm'
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(LM_DIR, exist_ok=True)
LM_PATH = os.path.join(LM_DIR, 'wita_train_4gram.pkl')


def reuse_or_run(out_path: str) -> bool:
    """
    Skip-if-exists guard.

    1. If `out_path` is already in /kaggle/working/, return True (skip).
    2. Otherwise look for a same-named file anywhere in /kaggle/input/**
       and copy it into /kaggle/working/.  Return True (skip).
    3. If nothing found, return False so the caller runs the eval.
    """
    if os.path.exists(out_path):
        print(f'[skip] {out_path} already exists -- reusing.')
        return True
    fname = os.path.basename(out_path)
    existing = _find(fname)
    if existing and existing != out_path:
        os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
        shutil.copy(existing, out_path)
        print(f'[skip] copied {existing} -> {out_path} (reusing).')
        return True
    return False

## Cell 3 — Train the 4-gram char LM on the FULL paper-split train labels

Subject-disjoint by construction (paper guarantees train signers don't appear in val/test).  <30 s on Kaggle CPU.

In [ ]:
if not reuse_or_run(LM_PATH):
    !python /kaggle/working/wita_v2/scripts/train_char_lm_stage11.py \
        --cache-root {CACHE_ROOT} --subsets lex nonlex \
        --order 4 --out {LM_PATH}
print(f'LM ready: {LM_PATH}')

## Cell 4 — Stage 9b sweep on VAL (CTC + LM prefix beam search)

Sweep `alpha ∈ {0.3, 0.5, 0.7, 1.0, 1.5}` × `beta ∈ {0.0, 0.5, 1.0}`.  15 configs × ~1 min each = ~15 min on T4.

In [ ]:
OUT_9b_SWEEP = os.path.join(LOG_DIR, 'stage11_9b_val_sweep.json')

if not reuse_or_run(OUT_9b_SWEEP):
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} --lm {LM_PATH} \
        --mode ctc_lm_beam --on val \
        --sweep 'ctc_lm_alpha=0.0,0.1,0.2,0.3,0.5' 'ctc_lm_beta=0.0,0.5' \
                'ctc_lm_beam=8' 'ctc_lm_symbol_top_k=10' \
        --out {OUT_9b_SWEEP}

with open(OUT_9b_SWEEP) as f:
    sweep_9b = json.load(f)
best_9b = min(sweep_9b, key=lambda r: r['overall_cer'])
print('\n9b val-best:', best_9b['hparams'], '=>', best_9b['overall_cer'])

## Cell 5 — Stage 9b TEST eval (once)

In [ ]:
OUT_9b_TEST = os.path.join(LOG_DIR, 'stage11_9b_test.json')

if not reuse_or_run(OUT_9b_TEST):
    hp_str = json.dumps(best_9b['hparams'])
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} --lm {LM_PATH} \
        --mode ctc_lm_beam --on test \
        --hparams '{hp_str}' --out {OUT_9b_TEST}

with open(OUT_9b_TEST) as f:
    print('9b test headline:', json.load(f))

## Cell 6 — B1 sweep on VAL (attention beam + length norm)

Sweep `beam_width ∈ {4, 8, 16}` × `length_alpha ∈ {0.5, 0.7, 0.9, 1.0}`.  12 configs × ~1.5 min each = ~18 min.

In [ ]:
OUT_B1_SWEEP = os.path.join(LOG_DIR, 'stage11_B1_val_sweep.json')

if not reuse_or_run(OUT_B1_SWEEP):
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} \
        --mode attn_beam --on val \
        --sweep 'beam_width=4,8,16' 'length_alpha=0.5,0.7,0.9,1.0' \
        --out {OUT_B1_SWEEP}

with open(OUT_B1_SWEEP) as f:
    sweep_B1 = json.load(f)
best_B1 = min(sweep_B1, key=lambda r: r['overall_cer'])
print('\nB1 val-best:', best_B1['hparams'], '=>', best_B1['overall_cer'])

## Cell 7 — B1 TEST eval (once)

In [ ]:
OUT_B1_TEST = os.path.join(LOG_DIR, 'stage11_B1_test.json')

if not reuse_or_run(OUT_B1_TEST):
    hp_str = json.dumps(best_B1['hparams'])
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} \
        --mode attn_beam --on test \
        --hparams '{hp_str}' --out {OUT_B1_TEST}

with open(OUT_B1_TEST) as f:
    print('B1 test headline:', json.load(f))

## Cell 8 — B2 sweep on VAL (joint CTC + attention, no LM)

Use 'joint' mode but set `alpha_lm=0` so this is CTC+attn only.  Sweep `alpha_ctc / alpha_attn` such that they sum to 1 (the canonical convention).

In [ ]:
import subprocess
OUT_B2_SWEEP = os.path.join(LOG_DIR, 'stage11_B2_val_sweep.json')

if not reuse_or_run(OUT_B2_SWEEP):
    # --- Resume-friendly: load best_B1 from Cell 6 if not in mem.
    if 'best_B1' not in globals():
        with open(os.path.join(LOG_DIR, 'stage11_B1_val_sweep.json')) as f:
            _b1 = json.load(f)
        best_B1 = min(_b1, key=lambda r: r['overall_cer'])
        print(f'[resume] best_B1 from disk: {best_B1["hparams"]}')

    B1_beam   = int(best_B1['hparams'].get('beam_width', 8))
    B1_lalpha = float(best_B1['hparams'].get('length_alpha', 0.7))
    pairs = [(0.2, 0.8), (0.3, 0.7), (0.5, 0.5), (0.7, 0.3), (0.8, 0.2)]
    results_B2 = []
    for ac, aa in pairs:
        hp = {'alpha_ctc': ac, 'alpha_attn': aa, 'alpha_lm': 0.0,
              'beam_width': B1_beam, 'length_alpha': B1_lalpha}
        out = f'/kaggle/working/logs/_b2_tmp_{ac}.json'
        if reuse_or_run(out):
            with open(out) as f:
                results_B2.extend(json.load(f)); continue
        cmd = ['python', '/kaggle/working/wita_v2/scripts/eval_stage11_decoders.py',
               '--cache-root', CACHE_ROOT, '--checkpoint', CKPT,
               '--mode', 'joint', '--on', 'val',
               '--hparams', json.dumps(hp), '--out', out]
        print(f'\n=== B2 config alpha_ctc={ac}, alpha_attn={aa} ===')
        proc = subprocess.run(cmd)
        if proc.returncode != 0 or not os.path.exists(out):
            print(f'  ⚠ B2 config ({ac}, {aa}) failed (rc={proc.returncode}); skipping.')
            continue
        with open(out) as f:
            results_B2.extend(json.load(f))
    with open(OUT_B2_SWEEP, 'w') as f:
        json.dump(results_B2, f, indent=2)

with open(OUT_B2_SWEEP) as f:
    results_B2 = json.load(f)
if results_B2:
    best_B2 = min(results_B2, key=lambda r: r['overall_cer'])
    print('\nB2 val-best:', best_B2['hparams'], '=>', best_B2['overall_cer'])
else:
    print('\n⚠ B2 sweep is empty. B2 test eval will be skipped.')
    best_B2 = None

## Cell 9 — B2 TEST eval (once)

In [ ]:
OUT_B2_TEST = os.path.join(LOG_DIR, 'stage11_B2_test.json')

if not reuse_or_run(OUT_B2_TEST):
    if 'best_B2' not in globals() or best_B2 is None:
        with open(os.path.join(LOG_DIR, 'stage11_B2_val_sweep.json')) as f:
            _b2 = json.load(f)
        if not _b2:
            raise RuntimeError('B2 sweep is empty -- cannot run B2 test.')
        best_B2 = min(_b2, key=lambda r: r['overall_cer'])
        print(f'[resume] best_B2 from disk: {best_B2["hparams"]}')
    hp_str = json.dumps(best_B2['hparams'])
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} \
        --mode joint --on test \
        --hparams '{hp_str}' --out {OUT_B2_TEST}

with open(OUT_B2_TEST) as f:
    print('B2 test headline:', json.load(f))

## Cell 10 — B3 combined sweep (KenLM + B1 + B2)

Initialise around the per-experiment optima from 9b / B1 / B2 and run a 16-config 4-axis grid.

In [ ]:
import itertools
OUT_B3_SWEEP = os.path.join(LOG_DIR, 'stage11_B3_val_sweep.json')

if not reuse_or_run(OUT_B3_SWEEP):
    if 'best_B1' not in globals():
        with open(os.path.join(LOG_DIR, 'stage11_B1_val_sweep.json')) as f:
            _b1 = json.load(f)
        best_B1 = min(_b1, key=lambda r: r['overall_cer'])
        print(f'[resume] best_B1 from disk: {best_B1["hparams"]}')
    if 'best_B2' not in globals() or best_B2 is None:
        with open(os.path.join(LOG_DIR, 'stage11_B2_val_sweep.json')) as f:
            _b2 = json.load(f)
        best_B2 = min(_b2, key=lambda r: r['overall_cer']) if _b2 else None
        if best_B2 is None:
            raise RuntimeError('best_B2 unavailable; cannot run B3.')
        print(f'[resume] best_B2 from disk: {best_B2["hparams"]}')

    results_B3 = []
    for ac, aa, alm, gl in itertools.product(
        [best_B2['hparams']['alpha_ctc'], 0.3, 0.5],
        [best_B2['hparams']['alpha_attn'], 0.3, 0.5],
        [0.3, 0.5, 1.0, 1.5],
        [0.0, 0.5],
    ):
        hp = {'alpha_ctc': ac, 'alpha_attn': aa, 'alpha_lm': alm,
              'gamma_len': gl,
              'beam_width': int(best_B1['hparams'].get('beam_width', 8)),
              'length_alpha': float(best_B1['hparams'].get('length_alpha', 0.7))}
        out = f'/kaggle/working/logs/_b3_tmp_{ac}_{aa}_{alm}_{gl}.json'
        if reuse_or_run(out):
            with open(out) as f:
                results_B3.extend(json.load(f)); continue
        cmd = ['python', '/kaggle/working/wita_v2/scripts/eval_stage11_decoders.py',
               '--cache-root', CACHE_ROOT, '--checkpoint', CKPT, '--lm', LM_PATH,
               '--mode', 'joint', '--on', 'val',
               '--hparams', json.dumps(hp), '--out', out]
        print(f'\n=== B3 config ac={ac} aa={aa} alm={alm} gl={gl} ===')
        proc = subprocess.run(cmd)
        if proc.returncode != 0 or not os.path.exists(out):
            print(f'  ⚠ config failed (rc={proc.returncode}); skipping.')
            continue
        with open(out) as f:
            results_B3.extend(json.load(f))
    with open(OUT_B3_SWEEP, 'w') as f:
        json.dump(results_B3, f, indent=2)

with open(OUT_B3_SWEEP) as f:
    results_B3 = json.load(f)
if results_B3:
    best_B3 = min(results_B3, key=lambda r: r['overall_cer'])
    print('\nB3 val-best:', best_B3['hparams'], '=>', best_B3['overall_cer'])
else:
    print('\n⚠ B3 sweep is empty.')
    best_B3 = None

## Cell 11 — B3 TEST eval (once)  — THIS IS THE HEADLINE

In [ ]:
OUT_B3_TEST = os.path.join(LOG_DIR, 'stage11_B3_test.json')

if not reuse_or_run(OUT_B3_TEST):
    if 'best_B3' not in globals() or best_B3 is None:
        with open(os.path.join(LOG_DIR, 'stage11_B3_val_sweep.json')) as f:
            _b3 = json.load(f)
        if not _b3:
            raise RuntimeError('B3 sweep is empty -- cannot run B3 test.')
        best_B3 = min(_b3, key=lambda r: r['overall_cer'])
        print(f'[resume] best_B3 from disk: {best_B3["hparams"]}')
    hp_str = json.dumps(best_B3['hparams'])
    !python /kaggle/working/wita_v2/scripts/eval_stage11_decoders.py \
        --cache-root {CACHE_ROOT} --checkpoint {CKPT} --lm {LM_PATH} \
        --mode joint --on test \
        --hparams '{hp_str}' --out {OUT_B3_TEST}

with open(OUT_B3_TEST) as f:
    print('B3 test headline:', json.load(f))

## Cell 12 — Compile the 5-row results table

In [ ]:
import os, json

# Resume-friendly: rebuild OUT_*_TEST paths if they weren't set in mem.
OUT_9b_TEST = os.path.join(LOG_DIR, 'stage11_9b_test.json')
OUT_B1_TEST = os.path.join(LOG_DIR, 'stage11_B1_test.json')
OUT_B2_TEST = os.path.join(LOG_DIR, 'stage11_B2_test.json')
OUT_B3_TEST = os.path.join(LOG_DIR, 'stage11_B3_test.json')

rows = []
rows.append({
    'config': 'Stage 11 baseline',
    'lex': 0.4348, 'nonlex': 0.5449, 'overall': 0.4498,
    'cer_1-4': 0.309, 'cer_13-inf': 0.662,
})
for name, path in [('+ Stage 9b (KenLM)',  OUT_9b_TEST),
                    ('+ B1 (length-norm beam)', OUT_B1_TEST),
                    ('+ B2 (joint CTC+attn)',   OUT_B2_TEST),
                    ('+ B3 (combined)',         OUT_B3_TEST)]:
    if not os.path.exists(path):
        print(f'⚠ Skipping row "{name}": {path} missing.')
        continue
    with open(path) as f:
        d = json.load(f)
    rows.append({
        'config':    name,
        'lex':       d['test_lex_cer'],
        'nonlex':    d['test_nonlex_cer'],
        'overall':   d['test_overall_cer'],
        'cer_1-4':   d['test_length_cer'].get('1-4',    float('nan')),
        'cer_13-inf':d['test_length_cer'].get('13-inf', float('nan')),
    })
rows.append({
    'config': 'Paper baseline (Kim et al. 2023)',
    'lex': 0.281, 'nonlex': 0.365, 'overall': 0.2924,
    'cer_1-4': '—', 'cer_13-inf': '—',
})
print(f"{'config':<38s}  {'lex':>7s}  {'nonlex':>7s}  {'overall':>7s}  {'1-4':>6s}  {'13+':>6s}")
for r in rows:
    print(f"{r['config']:<38s}  "
          f"{(r['lex']        if isinstance(r['lex'],     float) else r['lex']):>7}  "
          f"{(r['nonlex']     if isinstance(r['nonlex'],  float) else r['nonlex']):>7}  "
          f"{(r['overall']    if isinstance(r['overall'], float) else r['overall']):>7}  "
          f"{r['cer_1-4']:>6}  {r['cer_13-inf']:>6}")

with open('/kaggle/working/logs/stage11_decoder_fixes_table.json', 'w') as f:
    json.dump(rows, f, indent=2)

## Cell 13 — Commit kernel

Save Version → Save & Run All so the LM, results files, marker files, and table all survive.

Send `logs/stage11_decoder_fixes_table.json` here for the writeup and decision (band: STRONG_PASS / PASS / UNDERPERFORMS per §9).